# Stage 2 (uniform CTOA) - control experiment

Контрольный эксперимент: дообучаем `stage1_topo.pt` -> `stage2_topo_uniform.pt` на **uniform CTOA** (`Beta(1, 1)`) при сохранении всех остальных параметров Stage 2.

## Зачем

`stage2_topo.pt` обучен на CTOA `Beta(2.2, 1.6)` на [1000, 3300] ms - мода ~ 2540 ms. Pre-target tangling Q пик находится ровно у моды (bin 7 ~ 2700 ms), а не в середине диапазона. Возможны две интерпретации:

1. **Density artifact** - сеть просто чаще видела trial-ы у моды, через эту область проходит больше траекторий -> высокое Q там. Не вычисление.
2. **Функциональная структура** (hazard rate / preparation) - сеть учит готовность к таргету, и эта готовность зависит от выученной статистики времени.

При uniform CTOA hazard rate `h(t) = 1/(b-t)` монотонно растёт. Density плоская. Так что результат различает гипотезы:

| Результат при uniform | Интерпретация |
|---|---|
| Q плоское | Density artifact |
| Q монотонно растёт к концу | Hazard rate (функциональное) |
| Пик сохранился в той же позиции | Сеть запомнила Beta-форму (артефакт обучения) |

Параллельно проверяем: что станет с post-target Q vs CTOA (сейчас знак инвертирован относительно paper) и с PR-3 dimensionality.


In [ ]:
import sys, os, pathlib
ROOT = next(p for p in [pathlib.Path.cwd().resolve(), *pathlib.Path.cwd().resolve().parents]
            if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

from pathlib import Path
from collections import defaultdict
import numpy as np
import torch
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

from src.model_topo import BioLeakyRNNTopo
from src.env import CuedTargetWithDistractorsV3
from src.training import TrainConfig, train_supervised
from src.analysis import (
    collect_trials, filter_trials,
    tangling_by_ctoa_bin, prepare_jpca_input,
    participation_ratio, polynomial_regression,
    pr_singletrial_by_ctoa_bin,
)
from src.plotting import plot_tangling_vs_ctoa, plot_tangling_vs_rt

device = "cpu"
print("device:", device)
Path("checkpoints").mkdir(exist_ok=True)

from src.figsave import autosave
autosave("12_stage2_uniform_test")


## 1. Model factory & uniform-CTOA env factory

Архитектура - копия `make_model()` из `01c_train_topo.ipynb`. Env-фактори отличается только `ctoa_beta=(1.0, 1.0)`.


In [ ]:
def make_model():
    return BioLeakyRNNTopo(
        input_size=7,
        hidden_size=180,
        output_size=2,
        dt=20.0,
        tau=100.0,
        activation="softplus",
        sigma_rec=0.10,
        rec_init="diag",
        use_ei=True,
        exc_ratio=0.80,
        use_dale=True,
        mask_seed=42,
        sheet_side=12,
        tau_ee=0.25,
        tau_ie=0.32,
        tau_ei=0.64,
        tau_ii=0.64,
        rf_sigma=0.3,
        tau_e_range=(50, 150),
        tau_i_range=(10, 50),
        use_adaptation=False,
        tau_adapt=100.0,
        g_adapt=0.5,
    ).to(device)


def make_env_stage2_uniform():
    return CuedTargetWithDistractorsV3(
        dt=20,
        cue_strength=1.0,
        p_distractor_trial=0.6,
        distractor_strength=1.0,
        continuous_locations=True,
        ctoa_beta=(1.0, 1.0),
    )


# Sanity: check that distribution is actually uniform
env = make_env_stage2_uniform()
samples = [env._sample_ctoa_ms() for _ in range(10000)]
fig, ax = plt.subplots(figsize=(6, 3))
ax.hist(samples, bins=30, color="steelblue", edgecolor="black", linewidth=0.4)
ax.set_xlabel("CTOA (ms)")
ax.set_ylabel("count")
ax.set_title(f"Uniform sampling sanity: mean={np.mean(samples):.0f}, std={np.std(samples):.0f}")
plt.tight_layout(); plt.show()


## 2. Train Stage 2 from `stage1_topo.pt`

Если чекпоинт уже существует - пропускаем тренировку, грузим с диска. Чтобы переобучить заново, удалите файл `../checkpoints/stage2_topo_uniform.pt`.


In [ ]:
ckpt_out = Path("checkpoints/stage2_topo_uniform.pt")
history = None

if ckpt_out.exists():
    print(f"checkpoint exists: {ckpt_out} - skipping training")
else:
    model = make_model()
    model.load_state_dict(
        torch.load("checkpoints/stage1_topo.pt", weights_only=True)["state_dict"],
        strict=False,
    )
    cfg2 = TrainConfig(
        batch_size=64,
        lr=1e-3,
        max_updates=15000,
        print_every=50,
        device=device,
        fa_penalty_weight=0.0,
        com_weight=0.5,
    sparsity_weight=0.01,
    )  # default early stopping
    history = train_supervised(model, make_env_stage2_uniform, cfg2)
    torch.save({"state_dict": model.state_dict()}, ckpt_out)
    print(f"Saved: {ckpt_out}")


In [ ]:
# Training curves (only if just trained)
if history is not None:
    def smooth(x, w=20):
        x = np.asarray(x)
        if len(x) < w: return x
        return np.convolve(x, np.ones(w)/w, mode="valid")

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    upd = np.arange(1, len(history["p_correct"]) + 1)
    sw = 20
    for key, color, lab in [("p_correct","green","correct"),
                             ("p_miss","orange","miss"),
                             ("p_abort","red","abort")]:
        v = history[key]
        axes[0].plot(upd, v, alpha=0.15, color=color, lw=0.8)
        if len(v) >= sw:
            axes[0].plot(upd[sw-1:], smooth(v, sw), color=color, lw=2, label=lab)
    axes[0].set_ylim(-0.05, 1.05); axes[0].set_xlabel("update")
    axes[0].set_title("Stage 2 uniform - outcomes"); axes[0].legend(); axes[0].grid(alpha=0.3)

    axes[1].plot(upd, history["loss"], alpha=0.15, color="steelblue", lw=0.8)
    if len(history["loss"]) >= sw:
        axes[1].plot(upd[sw-1:], smooth(history["loss"], sw), color="steelblue", lw=2, label="total")
    axes[1].plot(upd, history["ce"], alpha=0.15, color="tomato", lw=0.8)
    if len(history["ce"]) >= sw:
        axes[1].plot(upd[sw-1:], smooth(history["ce"], sw), color="tomato", lw=2, label="CE")
    axes[1].set_xlabel("update"); axes[1].set_title("Stage 2 uniform - loss")
    axes[1].legend(); axes[1].grid(alpha=0.3)
    plt.tight_layout(); plt.show()
else:
    print("history is None (checkpoint loaded from disk)")


## 3. Collect trials + outcome distribution

Собираем 2000 trial-ей с обученной моделью на uniform-env (распределение CTOA-bin должно быть ровным).


In [ ]:
def make_eval_env():
    # IMPORTANT: use uniform env for analysis -> equal n trials per bin
    return make_env_stage2_uniform()

model = make_model()
model.load_state_dict(
    torch.load(ckpt_out, weights_only=True)["state_dict"], strict=False,
)
model.eval()
model.noise_at_eval = True  # CRITICAL for trial-to-trial diversity

torch.manual_seed(0); np.random.seed(0)
trials = collect_trials(model, make_eval_env, n_trials=2000, device=device)
corr   = filter_trials(trials, outcome="correct")

# Outcome breakdown
outcomes = defaultdict(int)
for tr in trials:
    outcomes[tr.get("train_outcome", "?")] += 1
print("Outcome distribution:")
for k, v in sorted(outcomes.items()):
    print(f"  {k:<10} {v:>5}  ({100*v/len(trials):.1f}%)")
print(f"\nCorrect trials: {len(corr)}/{len(trials)}")

# CTOA bin counts
bins = defaultdict(int)
for tr in corr:
    bins[tr["ctoa_bin"]] += 1
print(f"\nCTOA bin counts (correct): {[bins[i] for i in range(10)]}")


## 4. Tangling vs CTOA

- Pre-target window: -300..0 ms
- Post-target window: 0..+600 ms
- PCA pre-reduction до 20 dim


In [ ]:
tang_pre = tangling_by_ctoa_bin(
    corr, align_key="target_onset",
    window_before=15, window_after=0,
    pca_dims=20, outcome="correct", min_trials=10,
)
tang_post = tangling_by_ctoa_bin(
    corr, align_key="target_onset",
    window_before=0, window_after=10,
    pca_dims=20, outcome="correct", min_trials=10,
)

print("Pre-target  Q_mean per bin:", np.round(tang_pre["Q_mean"], 4))
print("Post-target Q_mean per bin:", np.round(tang_post["Q_mean"], 4))
pre_peak  = int(np.argmax(tang_pre["Q_mean"]))
post_peak = int(np.argmax(tang_post["Q_mean"]))
print(f"\nPre-target  peak bin: {pre_peak}  (CTOA={tang_pre['ctoa_ms_mean'][pre_peak]:.0f} ms)")
print(f"Post-target peak bin: {post_peak}  (CTOA={tang_post['ctoa_ms_mean'][post_peak]:.0f} ms)")


In [ ]:
# Dual y-axes: pre quadratic + post linear
plot_tangling_vs_ctoa(tang_pre, tang_post)


In [ ]:
# Linear vs quadratic fit comparison
print("Pre-target fits:")
for deg in [1, 2]:
    reg = polynomial_regression(tang_pre["ctoa_ms_mean"], tang_pre["Q_mean"], degree=deg)
    n = len(tang_pre["ctoa_ms_mean"]); k = deg + 1
    ss_res = np.sum((reg["y"] - reg["y_hat"]) ** 2) if reg["y_hat"] is not None else np.nan
    aic = n * np.log(ss_res / n) + 2 * k if np.isfinite(ss_res) and ss_res > 0 else np.nan
    coefs = np.round(reg["coeffs"], 6) if reg["coeffs"] is not None else None
    print(f"  deg-{deg}: R2={reg['r2']:.3f}  p={reg['p_value']:.4f}  AIC={aic:.1f}  coeffs={coefs}")

print("\nPost-target fits:")
for deg in [1, 2]:
    reg = polynomial_regression(tang_post["ctoa_ms_mean"], tang_post["Q_mean"], degree=deg)
    n = len(tang_post["ctoa_ms_mean"]); k = deg + 1
    ss_res = np.sum((reg["y"] - reg["y_hat"]) ** 2) if reg["y_hat"] is not None else np.nan
    aic = n * np.log(ss_res / n) + 2 * k if np.isfinite(ss_res) and ss_res > 0 else np.nan
    coefs = np.round(reg["coeffs"], 6) if reg["coeffs"] is not None else None
    print(f"  deg-{deg}: R2={reg['r2']:.3f}  p={reg['p_value']:.4f}  AIC={aic:.1f}  coeffs={coefs}")


## 5. RT vs CTOA + Tangling vs RT correlation

Ожидание: longer CTOA -> less tangling -> faster RT. У baseline (Beta-CTOA) у нас rho = -0.78. Смотрим, что в uniform.


In [ ]:
rt_by_bin = defaultdict(list)
for tr in corr:
    b  = tr.get("ctoa_bin")
    rt = tr.get("rt_ms")
    if b is not None and rt is not None:
        rt_by_bin[b].append(rt)

bins_lab = tang_post["labels"]
rt_mean = np.array([np.mean(rt_by_bin[b]) if rt_by_bin[b] else np.nan for b in bins_lab])
ctoa_x  = tang_post["ctoa_ms_mean"]

print("RT per CTOA bin:")
for b in bins_lab:
    n = len(rt_by_bin[b])
    if n:
        print(f"  bin {b}: {np.mean(rt_by_bin[b]):.1f} ms  (n={n})")


In [ ]:
rt_mean_dict = {b: rt_mean[i] for i, b in enumerate(tang_post["labels"])}
fig, rho, p = plot_tangling_vs_rt(tang_post, rt_mean_dict)
print(f"\nSpearman rho(Q_post, RT) = {rho:+.3f}, p = {p:.4f}")


In [ ]:
# RT vs CTOA - linear regression
from scipy import stats as sp_stats
mask = np.isfinite(ctoa_x) & np.isfinite(rt_mean)
slope, intercept, r, p_val, _ = sp_stats.linregress(ctoa_x[mask], rt_mean[mask])
print(f"RT vs CTOA: slope={slope:+.4f} ms/ms  R2={r**2:.3f}  p={p_val:.4f}")


## 6. Effective dimensionality (PR-3) vs CTOA

PR на 3 главных компонентах в pre-target окне (как в `10_tangling_dim_evolution.ipynb`).


In [ ]:
def pr_paperstyle_by_ctoa_bin(trials, align_key="target_onset",
                              window_before=15, window_after=0,
                              outcome="correct", n_components=3):
    """PR per CTOA bin restricted to top-n_components shared PCs."""
    X_cond, labels, rel_time, counts = prepare_jpca_input(
        trials, align_key=align_key,
        window_before=window_before, window_after=window_after,
        min_trials=5, outcome=outcome,
    )
    if X_cond is None:
        return None
    C, T, N = X_cond.shape
    X_flat = X_cond.reshape(C * T, N)
    k = min(n_components, N, X_flat.shape[0])
    pca = PCA(n_components=k).fit(X_flat)
    PR = []
    for c in range(C):
        Z = pca.transform(X_cond[c])
        PR.append(participation_ratio(Z))
    return {
        "PR":           np.array(PR),
        "labels":       labels,
        "rel_time":     rel_time,
        "counts":       counts,
        "n_components": k,
        "explained":    float(pca.explained_variance_ratio_.sum()),
    }


def ctoa_ms_mean(trials, labels, outcome="correct"):
    bin_to_ms = defaultdict(list)
    for tr in trials:
        if tr.get("train_outcome") != outcome:
            continue
        b, ms = tr.get("ctoa_bin"), tr.get("ctoa_ms")
        if b in labels and ms is not None:
            bin_to_ms[b].append(ms)
    return np.array([np.mean(bin_to_ms[b]) if b in bin_to_ms else np.nan
                      for b in labels])


def quad_fit(x, y):
    """Returns (coeffs, x_smooth, y_smooth, R2). NaN-safe."""
    x = np.asarray(x, dtype=float); y = np.asarray(y, dtype=float)
    m = np.isfinite(x) & np.isfinite(y)
    if m.sum() < 3:
        return None
    p = np.polyfit(x[m], y[m], 2)
    yhat = np.polyval(p, x[m])
    ss_res = np.sum((y[m] - yhat) ** 2)
    ss_tot = np.sum((y[m] - y[m].mean()) ** 2)
    R2 = 1 - ss_res / ss_tot if ss_tot > 0 else float("nan")
    xs = np.linspace(x[m].min(), x[m].max(), 200)
    return p, xs, np.polyval(p, xs), R2


In [ ]:
pr3 = pr_singletrial_by_ctoa_bin(
    corr, window_before=15, window_after=0, n_components=3,
)
pr3["ctoa_ms_mean"] = ctoa_ms_mean(corr, pr3["labels"])

print(f"PR-3 explained variance: {pr3['explained']:.3f}")
print(f"PR per bin: {np.round(pr3['PR'], 4)}")
pk = int(np.argmax(pr3["PR"]))
print(f"Peak bin:   {pk}  (CTOA={pr3['ctoa_ms_mean'][pk]:.0f} ms)")

fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(pr3["ctoa_ms_mean"], pr3["PR"], s=55, color="#ed7d31",
           edgecolor="k", linewidth=0.5, zorder=3)
fit = quad_fit(pr3["ctoa_ms_mean"], pr3["PR"])
if fit is not None:
    p, xs, ys, R2 = fit
    ax.plot(xs, ys, "--", color="red", lw=1.5, zorder=2)
    ax.text(0.62, 0.85, f"R^2 = {R2:.2f}", transform=ax.transAxes,
            color="red", fontsize=11)
    print(f"Quadratic fit: a={p[0]:+.3e}  b={p[1]:+.3e}  c={p[2]:+.3e}  R2={R2:.3f}")
    print(f"  -> inverted-U iff a<0 (a={p[0]:+.3e})")
ax.set_xlabel("CTOA (ms)"); ax.set_ylabel("Dimensionality (PR on 3 PCs)")
ax.set_title("Stage 2 uniform: PR-3 vs CTOA")
ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()


## 7. Side-by-side: uniform vs Beta baseline

Прогоняем тот же analysis на обеих моделях через **одинаковую** uniform-среду (равномерное n trial per bin) и собираем сравнительную таблицу.


In [ ]:
def run_full_analysis(ckpt_path, n_trials=2000):
    m = make_model()
    m.load_state_dict(torch.load(ckpt_path, weights_only=True)["state_dict"], strict=False)
    m.eval(); m.noise_at_eval = True
    torch.manual_seed(0); np.random.seed(0)
    trs  = collect_trials(m, make_eval_env, n_trials=n_trials, device=device)
    corr = filter_trials(trs, outcome="correct")
    if len(corr) < 30:
        return None
    pre  = tangling_by_ctoa_bin(corr, window_before=15, window_after=0,
                                  pca_dims=20, outcome="correct", min_trials=10)
    post = tangling_by_ctoa_bin(corr, window_before=0,  window_after=10,
                                  pca_dims=20, outcome="correct", min_trials=10)
    pr   = pr_singletrial_by_ctoa_bin(corr, window_before=15, window_after=0, n_components=3)
    pr["ctoa_ms_mean"] = ctoa_ms_mean(corr, pr["labels"])

    rtbb = defaultdict(list)
    for tr in corr:
        if tr.get("rt_ms") is not None:
            rtbb[tr["ctoa_bin"]].append(tr["rt_ms"])
    rt_m = np.array([np.mean(rtbb[b]) if rtbb[b] else np.nan for b in post["labels"]])

    from scipy.stats import spearmanr
    mask = np.isfinite(post["Q_mean"]) & np.isfinite(rt_m)
    rho, p_rho = spearmanr(post["Q_mean"][mask], rt_m[mask]) if mask.sum() >= 3 else (np.nan, np.nan)

    return {
        "n_corr": len(corr), "n_total": len(trs),
        "pre": pre, "post": post, "pr3": pr,
        "rt_mean": rt_m, "rho_post_rt": rho, "p_rho": p_rho,
    }


print("Running analysis on Beta baseline (stage2_topo.pt) ...")
res_beta = run_full_analysis("checkpoints/stage2_topo.pt")
print(f"  n_corr = {res_beta['n_corr']}/{res_beta['n_total']}")

print("Running analysis on uniform (stage2_topo_uniform.pt) ...")
res_uni  = run_full_analysis("checkpoints/stage2_topo_uniform.pt")
print(f"  n_corr = {res_uni['n_corr']}/{res_uni['n_total']}")


In [ ]:
# Combined plots: uniform (orange) vs Beta (blue)
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))

for ax, key, ylab, title in [
    (axes[0], "pre",  "Q_mean", "Pre-target tangling"),
    (axes[1], "post", "Q_mean", "Post-target tangling"),
    (axes[2], "pr3",  "PR",     "Dimensionality (PR-3)"),
]:
    for res, color, lab in [(res_beta, "#1f77b4", "Beta baseline"),
                             (res_uni,  "#ed7d31", "Uniform")]:
        if res is None: continue
        d = res[key]
        x = d["ctoa_ms_mean"]; y = d.get("Q_mean", d.get("PR"))
        ax.scatter(x, y, s=55, color=color, edgecolor="k", linewidth=0.5, zorder=3, label=lab)
        fit = quad_fit(x, y)
        if fit is not None:
            _, xs, ys, R2 = fit
            ax.plot(xs, ys, "--", color=color, lw=1.5, zorder=2, alpha=0.8)
    ax.set_xlabel("CTOA (ms)"); ax.set_ylabel(ylab); ax.set_title(title)
    ax.legend(fontsize=9); ax.grid(alpha=0.3)

plt.tight_layout(); plt.show()


In [ ]:
# Comparative summary table
def summarize(res, label):
    pre_fit  = quad_fit(res["pre"]["ctoa_ms_mean"],  res["pre"]["Q_mean"])
    pr_fit   = quad_fit(res["pr3"]["ctoa_ms_mean"],  res["pr3"]["PR"])

    pre_peak  = int(np.argmax(res["pre"]["Q_mean"]))
    pre_peak_ms = res["pre"]["ctoa_ms_mean"][pre_peak]
    pr_peak   = int(np.argmax(res["pr3"]["PR"]))
    pr_peak_ms = res["pr3"]["ctoa_ms_mean"][pr_peak]

    reg = polynomial_regression(res["post"]["ctoa_ms_mean"], res["post"]["Q_mean"], degree=1)
    post_slope = reg["coeffs"][0] if reg["coeffs"] is not None else float("nan")

    return {
        "label": label,
        "n_corr": res["n_corr"],
        "pre_R2":  pre_fit[3]  if pre_fit  else float("nan"),
        "pre_a":   pre_fit[0][0] if pre_fit  else float("nan"),
        "pre_peak_ms": pre_peak_ms,
        "post_slope": post_slope,
        "post_R2":   reg["r2"],
        "pr_R2":   pr_fit[3]  if pr_fit  else float("nan"),
        "pr_a":    pr_fit[0][0] if pr_fit  else float("nan"),
        "pr_peak_ms": pr_peak_ms,
        "rho_post_rt": res["rho_post_rt"],
    }


rows = []
if res_beta is not None: rows.append(summarize(res_beta, "Beta baseline"))
if res_uni  is not None: rows.append(summarize(res_uni,  "Uniform"))
print(f"{'Metric':<25} {'Beta':>16} {'Uniform':>16}")
print("-" * 60)
def fmt(v, w=16, p=4):
    if isinstance(v, str): return f"{v:>{w}}"
    if isinstance(v, (int, np.integer)): return f"{int(v):>{w}}"
    if not np.isfinite(v): return f"{'nan':>{w}}"
    if abs(v) < 1e-3 and v != 0: return f"{v:>{w}.2e}"
    return f"{v:>{w}.{p}f}"

keys = [("n_corr","n_corr"),
        ("Pre-Q  R2 (deg2)","pre_R2"), ("Pre-Q  a coef","pre_a"),
        ("Pre-Q  peak (ms)","pre_peak_ms"),
        ("Post-Q slope","post_slope"), ("Post-Q R2 (deg1)","post_R2"),
        ("PR-3   R2 (deg2)","pr_R2"), ("PR-3   a coef","pr_a"),
        ("PR-3   peak (ms)","pr_peak_ms"),
        ("rho(Q_post,RT)","rho_post_rt")]
for label, k in keys:
    line = f"{label:<25}"
    for r in rows:
        line += " " + fmt(r.get(k, "-"))
    print(line)


## 8. Интерпретация

Заполнить после прогона. Ключевые вопросы:

### A. Pre-target Q пик
- Если в **uniform** пик `Pre-Q` остался около той же CTOA-миллисекунды, что в Beta -> сеть запомнила Beta-форму распределения (артефакт выученной структуры обучения)
- Если пик **исчез** (Q стало плоским) -> density artifact, сеть просто реагировала на частые моменты в Beta
- Если Q стало **монотонно расти** к концу диапазона -> сеть учит hazard rate (1/(b-t))
- Если пик **сдвинулся в середину** -> есть какая-то функциональная схема preparation, не связанная ни с density, ни с hazard

### B. Post-target Q vs CTOA (sign)
- Если slope в uniform изменился с **+** на **-** -> знак инверсии в Beta был связан с распределением CTOA
- Если slope остался **+** -> расхождение с paper методологическое (вероятнее всего - окно `window_after=30` захватывает саккаду) либо архитектурное

### C. rho(Q_post, RT)
- Beta: -0.78. Если в uniform знак стал положительным -> проблема была в RT-CTOA связи

### D. Dimensionality
- Если PR в uniform по-прежнему ~ 1.000-1.004 -> низкая размерность не зависит от распределения, она структурная (архитектурная). Решается шумом / разными tau / расширением условий
- Если PR увеличился -> распределение влияло на компрессию динамики

После заполнения - план следующего шага зависит от результатов: возможно retrain с большим шумом, или урезать post-window, или менять архитектуру.
